# Cabana tutorial

This notebook shows the three entry points of the Cabana Python API on the sample image bundled with the repository. Run it from a clone of the repository with the `cabana` environment active (`conda activate cabana`), or change the paths to your own data.

See the [documentation](https://cabana.readthedocs.io/en/latest/) for the parameters and the meaning of every read-out.

In [ ]:
import os
from pathlib import Path

import pandas as pd

REPO = Path.cwd()
while not (REPO / "cabana" / "default_params.yml").exists() and REPO.parent != REPO:
    REPO = REPO.parent
PARAMS = REPO / "cabana" / "default_params.yml"
IMAGE = REPO / "data" / "Picrocirius" / "540.vsi - 20x_BF multi-band_01Annotation (Ellipse) (Tumor)_0.tif"
OUT = REPO / "tutorial_output"
print(PARAMS.exists(), IMAGE.exists())

## 1. Analyse one image

`Cabana` runs the whole pipeline (segmentation, fibre detection, skeleton and orientation analysis, HDM and gap analysis) on a single image and writes the same output folders as a batch run.

In [ ]:
from cabana import Cabana

analyzer = Cabana(str(PARAMS), str(IMAGE), str(OUT / "single"), ignore_large=True)
ok = analyzer.run()
if ok:
    analyzer.export_results()
    display(analyzer.stats.T.rename(columns={0: "value"}))
else:
    print("The image was skipped (too dark or too small).")

The result images are in `tutorial_output/single/`: `ROIs` (segmentation), `Fibres` (overlay), `Exports/<image>/` (width, skeleton, orientation and gap maps) and `Colors/<image>/` (visualisations).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

stem = IMAGE.stem
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, path) in zip(axes, [
        ("Input", OUT / "single" / "Eligible" / f"{stem}.png"),
        ("Detected fibres", OUT / "single" / "Fibres" / f"{stem}_fibres.png"),
        ("Gaps", OUT / "single" / "Masks" / "GapAnalysis" / f"{stem}_GapImage.png")]):
    ax.imshow(mpimg.imread(path)); ax.set_title(title); ax.axis("off")
plt.tight_layout()

## 2. Analyse a folder of images

`BatchProcessor` processes every image in a folder in batches with a checkpoint, merges the results into one `QuantificationResults.csv`, and can add per-patient statistics and collagen risk scores. Point `input_folder` at your images; `mask_dir` is optional (ROI masks named like the images, for example the `Masks` folder of a TMA export).

In [ ]:
from cabana import BatchProcessor

bp = BatchProcessor(
    batch_size=5,
    param_file=str(PARAMS),
    input_folder=str(IMAGE.parent),          # replace with your image folder
    output_folder=str(OUT / "batch"),
    generate_stats=True,
    generate_scores=False,
)
bp.status_callback = print
bp.run()
pd.read_csv(OUT / "batch" / "QuantificationResults.csv").head()

## 3. Cut a TMA slide into cores

`TMAPreprocessor` reads a whole-slide `.vsi` scan, fits every core, matches the printed ICGC/APGI array map and exports per-core images and circular masks named by patient. The exported `Images` and `Masks` folders are the input of a batch run (see the TMA tutorial in the documentation).

In [ ]:
from cabana import TMAPreprocessor

SLIDE = REPO / "large" / "APGI TMA" / "APGI TMA 4 PicRed.vsi"    # replace with your slide
if SLIDE.exists():
    pre = TMAPreprocessor(str(SLIDE), array_number=4, slide_name="TMA4")
    pre.fit()
    pre.map_to_array()
    print(len(pre.cores), "cores; orientation", pre.matched_orientation, "by", pre.orientation_method)
    if pre.orientation_resolved:
        pre.export(str(OUT / "TMA4_cores"), channels=["BF"])
    pre.close()
else:
    print("No slide at", SLIDE)